# Step 8 — CKD Model Comparison

**Project:** Hybrid Machine Learning Framework for Chronic Kidney Disease Prediction

### Objective
Compare the **actual Step 7 model-evaluation results** for all available algorithms using:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

This notebook **does not retrain models and does not perform hyperparameter tuning**. It only reads the results already generated in Step 7 and creates comparison tables and charts.

> This keeps Step 8 fast while preserving the complete comparison required for the project.


In [2]:
# ============================================================
# 1. Imports and locate project/results folder
# ============================================================
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

cwd = Path.cwd()

roots = [cwd, cwd.parent, cwd.parent.parent, cwd.parent.parent.parent]
project_root = next(
    (p for p in roots if (p / "dataset" / "raw" / "chronic_kidney_disease.csv").is_file()),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate project root. Expected dataset/raw/chronic_kidney_disease.csv"
    )

results_dir = project_root / "results"

print("Project root:", project_root)
print("Results folder:", results_dir)


Project root: c:\Users\Varshith\OneDrive\Desktop\KLH\2year\Machine Learning\Hybrid_ML_CKD_Prediction
Results folder: c:\Users\Varshith\OneDrive\Desktop\KLH\2year\Machine Learning\Hybrid_ML_CKD_Prediction\results


In [8]:
# # ============================================================
# # 2. Locate the Step 7 evaluation results
# # ============================================================
# candidates = [
#     results_dir / "CKD_Model_Evaluation_FAST.csv",
#     results_dir / "CKD_final_test_evaluation_after_tuning.csv",
#     results_dir / "CKD_tuning_and_evaluation_combined.csv",
# ]

# evaluation_file = next((p for p in candidates if p.exists()), None)

# if evaluation_file is None:
#     raise FileNotFoundError(
#         "No Step 7 evaluation CSV was found in the results folder.\n"
#         "Expected one of:\n" +
#         "\n".join(str(p) for p in candidates)
#     )

# print("Using Step 7 results:")
# print(evaluation_file)

# evaluation_df = pd.read_csv(evaluation_file)

# print("\nShape:", evaluation_df.shape)
# print("Columns:", evaluation_df.columns.tolist())
# display(evaluation_df)
# ============================================================
# 2. Locate the Step 7 evaluation results
# ============================================================
candidates = [
    results_dir / "CKD_Model_Evaluation_FAST.csv",
    results_dir / "CKD_final_test_evaluation_after_tuning.csv",
    results_dir / "CKD_tuning_and_evaluation_combined.csv",
    project_root / "source_dataset" / "results" / "dashboard_model_results.csv",
]

evaluation_file = next((p for p in candidates if p.exists()), None)

if evaluation_file is None:
    raise FileNotFoundError(
        "No Step 7 evaluation CSV was found in the results folder.\n"
        "Expected one of:\n" + 
        "\n".join(str(p) for p in candidates)
    )

print("Using Step 7 results:")
print(evaluation_file)

evaluation_df = pd.read_csv(evaluation_file)

print("\nShape:", evaluation_df.shape)
print("Columns:", evaluation_df.columns.tolist())
display(evaluation_df)



Using Step 7 results:
c:\Users\Varshith\OneDrive\Desktop\KLH\2year\Machine Learning\Hybrid_ML_CKD_Prediction\source_dataset\results\dashboard_model_results.csv

Shape: (9, 9)
Columns: ['Model', 'Training Accuracy', 'Training Precision', 'Training Recall', 'Training F1-Score', 'Training ROC-AUC', 'CV Mean Accuracy', 'CV Std', 'Generalization Gap']


,Model,Training Accuracy,Training Precision,Training Recall,Training F1-Score,Training ROC-AUC,CV Mean Accuracy,CV Std,Generalization Gap
0,SVM,1.000000,1.000000,1.0,1.000000,1.000000,0.996875,0.006250,0.003125
1,Logistic Regression,1.000000,1.000000,1.0,1.000000,1.000000,0.993750,0.007655,0.006250
2,Random Forest,1.000000,1.000000,1.0,1.000000,1.000000,0.990625,0.007655,0.009375
3,AdaBoost,1.000000,1.000000,1.0,1.000000,1.000000,0.990625,0.012500,0.009375
4,LightGBM,1.000000,1.000000,1.0,1.000000,1.000000,0.984375,0.017116,0.015625
5,XGBoost,0.996875,0.995025,1.0,0.997506,1.000000,0.981250,0.018222,0.015625
6,Gradient Boosting,1.000000,1.000000,1.0,1.000000,1.000000,0.975000,0.027243,0.025000
7,KNN,1.000000,1.000000,1.0,1.000000,1.000000,0.962500,0.012500,0.037500
8,Decision Tree,0.990625,0.985222,1.0,0.992556,0.999333,0.953125,0.017116,0.037500


In [9]:
# ============================================================
# 3. Standardize metric column names
# ============================================================
df = evaluation_df.copy()

# Map common column-name variations to standard names
rename_map = {}

for col in df.columns:
    c = col.strip().lower().replace("-", "_").replace(" ", "_")

    if c in ["accuracy", "test_accuracy"]:
        rename_map[col] = "Accuracy"
    elif c in ["precision", "test_precision"]:
        rename_map[col] = "Precision"
    elif c in ["recall", "test_recall"]:
        rename_map[col] = "Recall"
    elif c in ["f1", "f1_score", "test_f1", "test_f1_score"]:
        rename_map[col] = "F1"
    elif c in ["roc_auc", "roc_auc_score", "test_roc_auc"]:
        rename_map[col] = "ROC-AUC"

df = df.rename(columns=rename_map)

# Detect model column
model_col = next(
    (c for c in df.columns if c.lower() in ["model", "algorithm", "classifier", "model_name"]),
    None
)

if model_col is None:
    raise ValueError(
        "Could not identify the model-name column. Available columns: "
        + str(df.columns.tolist())
    )

print("Model column:", model_col)
print("Detected metrics:", [c for c in ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"] if c in df.columns])


Model column: Model
Detected metrics: []


In [29]:
# ============================================================
# 4. Build the final comparison table
# ============================================================
metrics = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
available_metrics = [m for m in metrics if m in df.columns]

comparison = df[[model_col] + available_metrics].copy()
comparison = comparison.rename(columns={model_col: "Model"})

# Convert metric values to numeric where possible
for metric in available_metrics:
    comparison[metric] = pd.to_numeric(comparison[metric], errors="coerce")

comparison = comparison.drop_duplicates(subset=["Model"]).reset_index(drop=True)

print("MODEL COMPARISON")
display(comparison.style.format(
    {m: "{:.4f}" for m in available_metrics}
))


MODEL COMPARISON


,Model
0,SVM
1,Logistic Regression
2,Random Forest
3,AdaBoost
4,LightGBM
5,XGBoost
6,Gradient Boosting
7,KNN
8,Decision Tree


In [30]:
# ============================================================
# 5. Check expected 9 algorithms
# ============================================================
expected_models = [
    "Logistic Regression",
    "Decision Tree",
    "KNN",
    "SVM",
    "Random Forest",
    "AdaBoost",
    "Gradient Boosting",
    "XGBoost",
    "LightGBM",
]

found_models = comparison["Model"].astype(str).tolist()

print("Expected algorithms:", len(expected_models))
print("Found algorithms:", len(found_models))

missing_models = [
    m for m in expected_models
    if not any(m.lower() == str(found).lower() for found in found_models)
]

if missing_models:
    print("\n⚠ Models not found in the Step 7 CSV:")
    for m in missing_models:
        print(" -", m)
else:
    print("\n✓ All 9 planned algorithms are present.")


Expected algorithms: 9
Found algorithms: 9

✓ All 9 planned algorithms are present.


In [28]:
# ============================================================
# 6. Accuracy comparison
# ============================================================
if "Accuracy" in comparison.columns:
    plt.figure(figsize=(10, 5))
    plt.bar(comparison["Model"], comparison["Accuracy"])
    plt.ylim(0, 1.05)
    plt.ylabel("Accuracy")
    plt.xlabel("Model")
    plt.title("CKD Model Comparison — Accuracy")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()



In [26]:
# ============================================================
# 7. Precision comparison
# ============================================================
if "Precision" in comparison.columns:
    plt.figure(figsize=(10, 5))
    plt.bar(comparison["Model"], comparison["Precision"])
    plt.ylim(0, 1.05)
    plt.ylabel("Precision")
    plt.xlabel("Model")
    plt.title("CKD Model Comparison — Precision")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


In [16]:
# ============================================================
# 8. Recall comparison
# ============================================================
if "Recall" in comparison.columns:
    plt.figure(figsize=(10, 5))
    plt.bar(comparison["Model"], comparison["Recall"])
    plt.ylim(0, 1.05)
    plt.ylabel("Recall")
    plt.xlabel("Model")
    plt.title("CKD Model Comparison — Recall")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


In [17]:
# ============================================================
# 9. F1-score comparison
# ============================================================
if "F1" in comparison.columns:
    plt.figure(figsize=(10, 5))
    plt.bar(comparison["Model"], comparison["F1"])
    plt.ylim(0, 1.05)
    plt.ylabel("F1-score")
    plt.xlabel("Model")
    plt.title("CKD Model Comparison — F1-score")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


In [18]:
# ============================================================
# 10. ROC-AUC comparison
# ============================================================
if "ROC-AUC" in comparison.columns:
    plt.figure(figsize=(10, 5))
    plt.bar(comparison["Model"], comparison["ROC-AUC"])
    plt.ylim(0, 1.05)
    plt.ylabel("ROC-AUC")
    plt.xlabel("Model")
    plt.title("CKD Model Comparison — ROC-AUC")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# 11. Combined metric comparison
# ============================================================
plot_metrics = [m for m in metrics if m in comparison.columns]

for metric in plot_metrics:
    plt.figure(figsize=(10, 5))
    plt.plot(comparison["Model"], comparison[metric], marker="o")
    plt.ylim(0, 1.05)
    plt.ylabel(metric)
    plt.xlabel("Model")
    plt.title(f"CKD Models — {metric} Comparison")
    plt.xticks(rotation=45, ha="right")
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()
 


In [20]:
# ============================================================
# 12. Descriptive comparison — no ranking/winner claim
# ============================================================
print("Metric ranges across the evaluated models:")

for metric in available_metrics:
    values = comparison[metric].dropna()
    if len(values):
        print(
            f"{metric}: minimum={values.min():.4f}, "
            f"maximum={values.max():.4f}, "
            f"mean={values.mean():.4f}"
        )

print("\nThis section describes the observed results without selecting a political-style")
print("winner/ranking; model choice should be based on the project's stated objectives.")


Metric ranges across the evaluated models:

This section describes the observed results without selecting a political-style
winner/ranking; model choice should be based on the project's stated objectives.


In [21]:
# ============================================================
# 13. Save Step 8 comparison results
# ============================================================
output_dir = results_dir
output_dir.mkdir(parents=True, exist_ok=True)

comparison_csv = output_dir / "CKD_Step8_Model_Comparison.csv"
comparison.to_csv(comparison_csv, index=False)

print("Saved:", comparison_csv)


Saved: c:\Users\Varshith\OneDrive\Desktop\KLH\2year\Machine Learning\Hybrid_ML_CKD_Prediction\results\CKD_Step8_Model_Comparison.csv


## Step 8 — Discussion Template

Use the actual values generated above in the report.

### Suggested discussion points

- Compare the nine algorithms using Accuracy, Precision, Recall, F1-score and ROC-AUC.
- Identify models with similar performance rather than relying on a single metric.
- Discuss the difference between individual models and ensemble/boosting approaches.
- Consider recall carefully because missing a CKD case can be important in a medical classification context.
- Refer back to the Step 6 cross-validation results and Step 7 test-set evaluation when discussing generalization.
- The 100% scores observed on a particular 80-record test split should be reported as **dataset-specific test results**, not as proof of perfect clinical performance.
- The UCI dataset contains 400 records; independent and larger clinical datasets would be needed for real-world validation.

### Important

Step 8 does **not** change, retrain, or tune the models. It only compares the results already produced in Step 7.
